# DS2002 · Building and Publishing Your API

**Studio — 2026-12-02 · Fall 2026**  
**Class time:** 45 minutes

---

## Building and publishing your API

Your team's API is the part of the capstone that someone outside the class could use. It reads the SQLite database your pipeline built and answers questions about ACC teams in JSON. The PDF lists the routes it needs. Today you practice the pattern on a small invented database, run it inside this notebook, and then follow the same steps to put your own version on PythonAnywhere.

The second half of the hour is for your presentation outline. Each team records an eight-minute video presentation and submits it with the project on Friday, December 18.

### Part 1: A route that reads SQLite

The first cell writes a tiny database file. It has the same four invented teams as Monday's clinic, so you can see the whole table at once.

In [ ]:
import sqlite3

with sqlite3.connect('toy.db') as conn:
    conn.executescript('''
    DROP TABLE IF EXISTS teams;
    CREATE TABLE teams (team TEXT PRIMARY KEY, wins INTEGER, losses INTEGER,
                        avg_margin REAL);
    INSERT INTO teams VALUES ('Harbor', 3, 1, 6.5), ('Ridge', 3, 1, 4.0),
                             ('Summit', 1, 3, -2.8), ('Valley', 1, 3, -7.8);
    ''')
print('wrote toy.db')

FastAPI may not be installed in Colab or Kaggle, so install it first. `httpx` is what FastAPI's test client uses to send requests to the app without starting a server.

In [ ]:
%pip install -q fastapi httpx

The app opens a new connection for each request and closes it when the request is done. Setting `row_factory` to `sqlite3.Row` lets each row be turned into a dictionary, which FastAPI sends back as JSON. Notice that the route checks whether the team exists and returns a 404 error when it does not, instead of returning an empty answer that looks like data.

In [ ]:
import os
from fastapi import FastAPI, HTTPException

DB_PATH = os.path.abspath('toy.db')
app = FastAPI(title='Toy league')

def query(sql, params=()):
    with sqlite3.connect(DB_PATH) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(r) for r in conn.execute(sql, params).fetchall()]

@app.get('/teams')
def list_teams():
    return query('SELECT * FROM teams ORDER BY avg_margin DESC')

@app.get('/teams/{name}')
def one_team(name: str):
    rows = query('SELECT * FROM teams WHERE lower(team) = lower(?)', (name,))
    if not rows:
        raise HTTPException(status_code=404, detail=f'no team named {name}')
    return rows[0]

The question mark in the second query is a placeholder. SQLite fills it with the value you pass, which keeps a user's input from being read as part of the SQL. Build every query in your API this way, and never paste a request value into the SQL string with an f-string.

The test client calls the routes the same way a browser would.

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)
print(client.get('/teams').json())
print(client.get('/teams/ridge').json())
print(client.get('/teams/nowhere').status_code)

**Try it:** add a route `/leaders?stat=wins` that returns the teams sorted by whichever column the caller asks for. Because a column name cannot be passed through a `?` placeholder, check the value against a list of allowed columns and return a 400 error for anything else. Test it with the client, including one bad value.

In [ ]:
# TODO: the /leaders route and three test calls

### Part 2: Put your own API on PythonAnywhere

The deploy how-to in this folder walks through account setup and the `pa website create` command. These are the steps that are specific to an API that reads a database.

1. Keep `main.py`, `requirements.txt`, and a copy of `acc.db` together in one folder of your team repository, such as `api/`.
2. In `main.py`, build the database path from the location of the file itself, as in `os.path.join(os.path.dirname(__file__), 'acc.db')`. The server does not start in your folder, so a bare `'acc.db'` will not be found.
3. Upload `main.py` and `acc.db` to the folder your website command points to, using the Files tab or the API.
4. Install `fastapi` and `uvicorn` in the virtual environment named in your website command.
5. Reload the site, then open `/docs` and try every route.
6. Each time your pipeline produces a newer `acc.db`, upload it again and reload. Your API makes no outside requests, so the free account's limits on outside sites do not affect it.

If a route fails after a reload, the error log at `/var/log/USERNAME.pythonanywhere.com.error.log` usually names the line.

### Part 3: Call it from another notebook

One teammate who did not write the API should call it from their own notebook with `requests` and draw chart 4 from the JSON it returns. If that teammate cannot understand a response without asking the author, the route needs clearer field names. The cell below shows the call pattern against the class demo site.

In [ ]:
import requests

r = requests.get('https://ds2002jason.pythonanywhere.com/time', timeout=20)
print(r.status_code)
print(r.json())

In [ ]:
API_URL = 'TODO: https://YOURUSERNAME.pythonanywhere.com'

# TODO: call one of your own routes and print the status code and the first record

### Part 4: Outline the presentation

Your presentation is a recorded video of no more than eight minutes, in which every member speaks. A screen recording over your slides or your notebook works well, and the link goes in the README. Eight minutes goes quickly with four speakers. Write each part of the outline below as a complete sentence that states a result. If a sentence cannot be written yet, that part of the analysis is not finished, and knowing that today leaves time to finish it.

| Part | Time | What it needs to do |
|---|---|---|
| The question | 0.5 min | Explain to a non-fan what you set out to learn |
| The data and pipeline | 1.5 min | Name both sources and one cleaning decision you can defend |
| Findings | 4 min | Questions 1 through 6, each with its number and a chart |
| The API | 1 min | Show a live route and say who would use it |
| What broke | 1 min | One real failure and what you changed because of it |

**The question:** _..._

**The data and pipeline:** _..._

**Findings:** _..._

**The API:** _..._

**What broke:** _..._

### Team progress

Nothing in this cell is submitted or graded. Fill it in so your team can see what still needs to happen before the final submission on Friday, December 18.

In [ ]:
progress = {
    'api_url': 'TODO',                  # your live address, or 'not deployed yet'
    'routes_working': None,             # TODO: how many of the five required routes respond
    'toy_leaders_route_passed': False,  # TODO: True if your /leaders test calls passed
    'outline_parts_written': None,      # TODO: how many of the five parts are full sentences
    'blocker': 'TODO',
}

for k, v in progress.items():
    print(f'{k}: {v}')